# Getting started — Iceberg on SeaweedFS (Spark)

This notebook runs inside the `spark-iceberg` pod on the kind cluster. It talks to:

- **Iceberg REST catalog** at `http://iceberg-rest:8181` (catalog name: `demo`)
- **SeaweedFS** S3 storage at `http://seaweedfs:8333` (bucket `warehouse`)

The catalog config lives in `$SPARK_HOME/conf/spark-defaults.conf`, so a plain
`getOrCreate()` is already wired to the `demo` catalog.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("getting-started").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| default catalog:", spark.conf.get("spark.sql.defaultCatalog"))

## Create a namespace and an Iceberg table

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.nyc")
spark.sql("""
  CREATE TABLE IF NOT EXISTS demo.nyc.taxis (
    vendor_id   BIGINT,
    trip_id     BIGINT,
    trip_distance FLOAT,
    fare_amount DOUBLE,
    store_and_fwd_flag STRING
  )
  USING iceberg
  PARTITIONED BY (vendor_id)
""")
print("table created")

In [ ]:
spark.sql("""
  INSERT INTO demo.nyc.taxis VALUES
    (1, 1000371, 1.8, 15.32, 'N'),
    (2, 1000372, 2.5, 22.15, 'N'),
    (2, 1000373, 0.9,  9.01, 'N'),
    (1, 1000374, 8.4, 42.13, 'Y')
""")
print("rows inserted")

## Query it

Use the `%%sql` magic (renders a pretty table) or the DataFrame API.

In [ ]:
%%sql
SELECT vendor_id, count(*) AS trips, round(avg(fare_amount), 2) AS avg_fare
FROM demo.nyc.taxis
GROUP BY vendor_id
ORDER BY vendor_id

## Where does the data live?

Iceberg's metadata tables expose the physical files — note they sit in
`s3://warehouse/` on SeaweedFS.

In [ ]:
spark.sql("SELECT file_path, record_count FROM demo.nyc.taxis.files").show(truncate=False)

In [ ]:
spark.sql("SELECT committed_at, snapshot_id, operation FROM demo.nyc.taxis.snapshots").show(truncate=False)

## Bonus: same catalog from PyIceberg (no Spark)

`/root/.pyiceberg.yaml` is pre-configured, so this works out of the box.

In [ ]:
from pyiceberg.catalog import load_catalog

cat = load_catalog("default")
print("namespaces:", cat.list_namespaces())
print("tables in nyc:", cat.list_tables("nyc"))